In [36]:
import torch
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from helper_function import plot_decision_boundary, plot_loss_curves, plot_predictions


In [37]:
device = torch.accelerator.current_accelerator().type

In [38]:
device

'cuda'

In [ ]:
torch.set_default_device(device)

In [48]:
# Set the hyperparameters for data creation
NUM_CLASSES = 5
NUM_FEATURES = 5
RANDOM_SEED = 42

In [49]:
X_blob, y_blob = make_blobs(n_samples=1000,n_features=NUM_FEATURES,centers=NUM_CLASSES,random_state=RANDOM_SEED,cluster_std=1.5)

In [50]:
X_blob[:5]

array([[-7.15631668, -6.38354406,  2.75344642,  0.38577558, -4.30630122],
       [-0.6744412 ,  7.98913844,  3.87509247, -1.75374768, -5.49988593],
       [ 0.90444432,  9.62362616,  3.99196553,  4.76312019, -5.07900987],
       [-1.72326862,  8.13215589,  4.43977755,  3.57966438, -7.75382809],
       [ 3.67621869, -5.28204487, -5.09547799, -3.27041737,  0.98636802]])

In [61]:
X_blob_data = torch.from_numpy(X_blob).to(dtype=torch.float)
y_blob_data = torch.from_numpy(y_blob).to(dtype=torch.float)

In [62]:
X_blob_data.dtype,X_blob_data.shape

(torch.float32, torch.Size([1000, 5]))

In [70]:
X_train, X_test, y_train , y_test = train_test_split(X_blob_data,y_blob,test_size=0.2,random_state=42)

In [69]:
torch.cuda.manual_seed(42)
torch.manual_seed(42)

In [98]:
X_train[:2]

tensor([[ -8.1923, -12.2529,   7.7509,   2.3329,   5.8749],
        [  5.4546,  -5.8681,  -2.7028,  -1.9657,  -1.4296]])

In [95]:
import torch.nn as nn
from torch.nn import Sequential
from torch.nn import CrossEntropyLoss
from torch.optim import Adam


In [106]:
class MulticlassClassification(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(5,10)
        self.layer2 = nn.Linear(10,15)
        self.layer3 = nn.Linear(15,10)
        self.layer4 = nn.Linear(10,5)
        

    def forward(self,X:torch.Tensor)-> torch.Tensor:
        x = nn.ReLU(self.layer1(X))
        x = nn.ReLU(self.layer2(x))
        x = nn.ReLU(self.layer3(x))
        x = nn.ReLU(self.layer4(x))
        x = nn.Softmax(x)
        return x

In [100]:
model = MulticlassClassification()


In [102]:
list(model.parameters())

[Parameter containing:
 tensor([[ 0.1010, -0.4382, -0.0909, -0.4112, -0.3074],
         [-0.0157,  0.2113, -0.0841,  0.0169, -0.1908],
         [-0.2311,  0.3782,  0.2951, -0.4166, -0.1003],
         [-0.3735,  0.2452, -0.1975,  0.0124, -0.2622],
         [-0.4406,  0.3354,  0.4188, -0.3398, -0.2708],
         [-0.0445, -0.4015, -0.0097,  0.3874, -0.1067],
         [-0.4017, -0.2091,  0.1337, -0.2080, -0.2922],
         [ 0.3188, -0.2002, -0.2600, -0.3836,  0.2416],
         [ 0.0984, -0.1392,  0.4340, -0.0276,  0.3760],
         [ 0.1903, -0.3532,  0.2807, -0.3495,  0.0983]], device='cuda:0',
        requires_grad=True),
 Parameter containing:
 tensor([ 0.4362, -0.3319,  0.0555,  0.0198,  0.2187,  0.0854,  0.4156,  0.3559,
          0.2441,  0.1504], device='cuda:0', requires_grad=True),
 Parameter containing:
 tensor([[ 0.2061, -0.1816,  0.1450, -0.1132,  0.1613, -0.1814,  0.2706, -0.2250,
          -0.0966,  0.3146],
         [ 0.0336, -0.1587,  0.2721, -0.1396,  0.2168,  0.0135, -0

In [103]:
model.state_dict()

OrderedDict([('layer1.weight',
              tensor([[ 0.1010, -0.4382, -0.0909, -0.4112, -0.3074],
                      [-0.0157,  0.2113, -0.0841,  0.0169, -0.1908],
                      [-0.2311,  0.3782,  0.2951, -0.4166, -0.1003],
                      [-0.3735,  0.2452, -0.1975,  0.0124, -0.2622],
                      [-0.4406,  0.3354,  0.4188, -0.3398, -0.2708],
                      [-0.0445, -0.4015, -0.0097,  0.3874, -0.1067],
                      [-0.4017, -0.2091,  0.1337, -0.2080, -0.2922],
                      [ 0.3188, -0.2002, -0.2600, -0.3836,  0.2416],
                      [ 0.0984, -0.1392,  0.4340, -0.0276,  0.3760],
                      [ 0.1903, -0.3532,  0.2807, -0.3495,  0.0983]], device='cuda:0')),
             ('layer1.bias',
              tensor([ 0.4362, -0.3319,  0.0555,  0.0198,  0.2187,  0.0854,  0.4156,  0.3559,
                       0.2441,  0.1504], device='cuda:0')),
             ('layer2.weight',
              tensor([[ 0.2061, -0.1816,  0.1

In [104]:
optim_fn = Adam(model.parameters(),lr=0.01)

In [105]:
loss_fn = CrossEntropyLoss()